# Extra Credit Code
This is the code for the extra credits, that is Part 8 of the Coding Assessment 2. Here I am using A* algorithm for finding a path from initial given pose3 to the target pose3.

A* Algorithm: A* algorithm divides the whole 3D envirnment into grids. It then marks all the grid cells that have the obstacle in them. Finally by defining the cost function, it tries to reach the goal position with the least cost.

### Mounting on Drive!

In [134]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Importing necessary Libraries

In [135]:
## installation

## imports
from typing import List, Tuple, Dict, Optional, Set
import math
import numpy as np
import pandas as pd
import unittest
import plotly.express as px
import plotly.graph_objects as go
%pip install gtsam==4.3a0
import gtsam
import time
import heapq

rng = np.random.default_rng(12345)

print("All packages imported successfully!")

All packages imported successfully!


### Helper Functions

In [136]:
#downloads the helpers file. This can be commented after running once
!pip install --upgrade --no-cache-dir gdown &> /dev/null
!gdown 1rOEki1n8coTVURLBL7Yh2icbpPc-1fB_

import helpers_obstacles as helpers
# the Drone dynamics model is described in detail in: https://www.roboticsbook.org/S72_drone_actions.html
from helpers_obstacles import Drone, axes, axes_figure

print("Helper functions are here!")

Downloading...
From (original): https://drive.google.com/uc?id=1rOEki1n8coTVURLBL7Yh2icbpPc-1fB_
From (redirected): https://drive.google.com/uc?id=1rOEki1n8coTVURLBL7Yh2icbpPc-1fB_&confirm=t&uuid=b2f848f0-6c2b-4e53-9d1b-92b4abf233d8
To: /content/helpers_obstacles.py
100% 61.3k/61.3k [00:00<00:00, 49.8MB/s]
Helper functions are here!


### Environment setup code (Extra Credit Section)

In [137]:
## ENVIRONMENT SETUP - 8 OBSTACLE HARD COURSE

# Get the racing environment components
hoops = helpers.get_hoops()
targets = helpers.get_targets()
start_pose = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(1, 3, 8))

# Load HARD obstacle course (8 obstacles - mix of spheres and boxes)
obstacles_hard = helpers.get_obstacles_hard()

print("=" * 80)
print("🏁 DRONE RACING ENVIRONMENT - HARD COURSE")
print("=" * 80)
print(f"\n📍 Start Position: {start_pose.translation()}")
print(f"🎯 Number of Hoops: {len(hoops)}")
print(f"🚧 Number of Obstacles: {len(obstacles_hard)}")
print(f"\nObstacle Details:")
for i, obs in enumerate(obstacles_hard):
    if hasattr(obs, 'radius'):  # Sphere obstacle
        print(f"   Obstacle {i+1}: Sphere at {obs.center}, radius={obs.radius}")
    else:  # Box obstacle
        print(f"   Obstacle {i+1}: Box from {obs.min_corner} to {obs.max_corner}")

print("\n" + "=" * 80)

🏁 DRONE RACING ENVIRONMENT - HARD COURSE

📍 Start Position: [1. 3. 8.]
🎯 Number of Hoops: 4
🚧 Number of Obstacles: 8

Obstacle Details:
   Obstacle 1: Sphere at [3. 3. 7.], radius=1.2
   Obstacle 2: Sphere at [6. 4. 6.], radius=1.0
   Obstacle 3: Sphere at [8. 8. 3.], radius=1.5
   Obstacle 4: Sphere at [ 8. 11.  8.], radius=1.3
   Obstacle 5: Sphere at [ 6.5  8.  11. ], radius=1.0
   Obstacle 6: Sphere at [ 5.5  6.5 11.5], radius=0.8
   Obstacle 7: Box from [7.5 7.  0. ] to [9.  9.  1.5]
   Obstacle 8: Box from [4.  4.5 9. ] to [ 6.   5.5 12. ]



In [138]:
## Visualize the environment before planning

print("📊 Visualizing environment...")

# Show the racing course with obstacles (no path yet)
fig = helpers.drone_racing_path_with_obstacles(
    hoops=hoops,
    start=start_pose,
    path=[],  # Empty path for now
    obstacles=obstacles_hard
)

fig

print("\n✅ Environment visualization complete!")
print("🎯 Goal: Navigate through all 4 hoops while avoiding 8 obstacles")

📊 Visualizing environment...



✅ Environment visualization complete!
🎯 Goal: Navigate through all 4 hoops while avoiding 8 obstacles


## Functions from Coding Assessment
 These fucntions can be used here as it is for implementing A*. Thus directly copy-pasted for use. The codes and the To-Do numbers are mentioned.

In [139]:
# Importing necessary functions from the Coding Assessment

# TODO 6: Compute attitude from yaw, pitch, roll
def compute_attitude_from_ypr(yaw: float, pitch: float, roll: float) -> gtsam.Rot3:
    """
    Uses yaw, pitch and roll angles to compute the attitude of the drone

    Arguments:
        yaw: float (in radians)
        pitch: float (in radians)
        roll: float (in radians)

    Returns:
        attitude: gtsam.Rot3
    """
    attitude = gtsam.Rot3.RzRyRx(roll, pitch, yaw)
    return attitude


# TODO 7: Compute force vector
def compute_force(attitude: gtsam.Rot3, thrust: float) -> gtsam.Point3:
    """
    Computes the force vector given attitude and thrust in the body frame

    Arguments:
        attitude: gtsam.Rot3, nRb for the drone
        thrust: float, the upwards thrust produced by the 4 rotors

    Returns:
        force: gtsam.Point3, the resultant force vector
    """
    force = attitude.rotate(gtsam.Point3(0, 0, thrust))
    return force


# TODO 8: Compute terminal velocity
def compute_terminal_velocity(force: gtsam.Point3, kd: float = 0.0425) -> gtsam.Point3:
    """
    Uses the force vector and drag coefficient to compute the terminal velocity

    Arguments:
        force: gtsam.Point3, the force vector in the navigation frame
        kd: float, drag coefficient

    Returns:
        terminal_velocity: gtsam.Point3, the maximum velocity vector
    """
    eps = 1e-6
    force_values = [force[0], force[1], force[2]]
    signs = np.sign(force_values)
    force_values = np.sqrt(np.square(force_values))
    force_values = (force_values + eps) / kd
    force_values = signs * np.sqrt(force_values)

    terminal_velocity = gtsam.Point3(force_values[0], force_values[1], force_values[2])
    return terminal_velocity


# TODO 12: Compute force with gravity
def compute_force_with_gravity(attitude: gtsam.Rot3, thrust: float, mass: float = 1.0) -> gtsam.Point3:
    """
    Computes the net force vector given attitude and thrust by adjusting for gravity

    Arguments:
        attitude: gtsam.Rot3, nRb for the drone
        thrust: float, the upwards thrust produced by the 4 rotors
        mass: float, the mass of the drone, default: 1.0

    Returns:
        force: gtsam.Point3, the resultant force vector
    """
    g = 10.0  # m/s^2
    force = compute_force(attitude, thrust)
    force_array = np.array([force[0], force[1], force[2]])
    force_array[2] = force_array[2] - mass * g
    return gtsam.Point3(force_array[0], force_array[1], force_array[2])


print(" Helper functions loaded!")


 Helper functions loaded!


### A* Configuration Class
This class defines the parameters we will use for the A* Algorithm.

In [140]:
# A* Planner Configuration Class

class AStarConfig:
    #Configuration parameters for A* path planner

    # Grid settings
    GRID_RESOLUTION = 0.4  # meters per cell

    # Workspace bounds [min, max] for each axis
    WORKSPACE_X = (0.0, 10.0)
    WORKSPACE_Y = (0.0, 10.0)
    WORKSPACE_Z = (0.0, 10.0)

    # Safety margin around obstacles
    SAFETY_MARGIN = 0.2  # meters (keep drone 0.2m away from obstacles)

    # Movement costs
    COST_STRAIGHT = 1.0             # Moving along one axis
    COST_DIAGONAL_2D = np.sqrt(2)   # Moving along two axes
    COST_DIAGONAL_3D = np.sqrt(3)   # Moving along three axes

    # Neighbor connectivity
    USE_26_CONNECTED = True   # 3x3x3 = 27, Subtracting 1 (Current position)

    def __init__(self):
        # Calculate grid dimensions
        # Adding +1 to ensure we can reach max boundary
        self.nx = int(np.ceil((self.WORKSPACE_X[1] - self.WORKSPACE_X[0]) / self.GRID_RESOLUTION))
        self.ny = int(np.ceil((self.WORKSPACE_Y[1] - self.WORKSPACE_Y[0]) / self.GRID_RESOLUTION))
        self.nz = int(np.ceil((self.WORKSPACE_Z[1] - self.WORKSPACE_Z[0]) / self.GRID_RESOLUTION))

        print(f"Grid Configuration:")
        print(f"  Resolution: {self.GRID_RESOLUTION}m")
        print(f"  Dimensions: {self.nx} x {self.ny} x {self.nz} = {self.nx * self.ny * self.nz:,} cells")
        print(f"  Safety margin: {self.SAFETY_MARGIN}m")
        print(f"  Connectivity: {'26-connected' if self.USE_26_CONNECTED else '6-connected'}")

# Create configuration instance
config = AStarConfig()

Grid Configuration:
  Resolution: 0.4m
  Dimensions: 25 x 25 x 25 = 15,625 cells
  Safety margin: 0.2m
  Connectivity: 26-connected


### Grid Cell Class
This class gives the parameters for the grids that we divide our envirnment in!

In [141]:
# Grid Cell Class - Represents a single cell in the 3D grid

class GridCell:
    """
    Represents a single cell in the 3D A* search grid

    Attributes:
        i, j, k: Grid coordinates (integers)
        x, y, z: World coordinates (floats) - center of cell
        g_cost: Actual cost from start to this cell
        h_cost: Heuristic estimate from this cell to goal
        f_cost: Total cost (g + h)
        parent: Parent GridCell (for path reconstruction)
        is_obstacle: True if cell contains obstacle
    """

    def __init__(self, i: int, j: int, k: int, world_x: float, world_y: float, world_z: float):
        # Grid coordinates
        self.i = i
        self.j = j
        self.k = k

        # World coordinates
        self.x = world_x
        self.y = world_y
        self.z = world_z

        # A* costs
        self.g_cost = float('inf')
        self.h_cost = 0.0
        self.f_cost = float('inf')

        # Path reconstruction
        self.parent = None

        # Obstacle flag
        self.is_obstacle = False

    def __lt__(self, other):
        # For priority queue comparison - compare f_costs
        return self.f_cost < other.f_cost

    def __eq__(self, other):
        # Two cells are equal if they have same grid coordinates
        return (self.i == other.i and
                self.j == other.j and
                self.k == other.k)

    def __hash__(self):
        # For use in sets and dictionaries
        return hash((self.i, self.j, self.k))

    def position_tuple(self):
        # Return (i, j, k) as tuple for easy comparison
        return (self.i, self.j, self.k)

    def world_position(self):
        # Return (x, y, z) as numpy array
        return np.array([self.x, self.y, self.z])

    def __repr__(self):
        return f"Cell({self.i},{self.j},{self.k}) @ ({self.x:.1f},{self.y:.1f},{self.z:.1f})"


print("Grid Cell class created!")


Grid Cell class created!


### Frame Conversion Functions
These functions are used to convert the coordinates from given frame to the A* algorithm's grid frame. Basically the world frame has 1m as unit length, but grid has 0.4m as the unit distance.

In [142]:
# Coordinate conversion functions

def world_to_grid(world_pos: np.ndarray, config: AStarConfig) -> Tuple[int, int, int]:
    """
    Convert world coordinates (x, y, z) to grid indices (i, j, k)

    Arguments:
        world_pos: numpy array [x, y, z] in meters
        config: AStarConfig with grid parameters

    Returns:
        (i, j, k): Grid cell indices (integers)

    """

    # Extract x, y, z from world_pos
    x, y, z = world_pos[0], world_pos[1], world_pos[2]

    # Convert to grid indices
    # i = (x - x_min) / resolution, rounded down
    i = int((x - config.WORKSPACE_X[0]) / config.GRID_RESOLUTION)
    j = int((y - config.WORKSPACE_Y[0]) / config.GRID_RESOLUTION)
    k = int((z - config.WORKSPACE_Z[0]) / config.GRID_RESOLUTION)

    # Clamp to valid range [0, n-1] to prevent out-of-bounds errors
    i = np.clip(i, 0, config.nx - 1)
    j = np.clip(j, 0, config.ny - 1)
    k = np.clip(k, 0, config.nz - 1)
    return (i, j, k)


def grid_to_world(i: int, j: int, k: int, config: AStarConfig) -> np.ndarray:
    """
    Convert grid indices (i, j, k) to world coordinates (x, y, z)
    Returns the CENTER of the cell

    Arguments:
        i, j, k: Grid cell indices (integers)
        config: AStarConfig with grid parameters

    Returns:
        world_pos: numpy array [x, y, z] at cell center

    """

    # Add 0.5 to get cell center, then multiply by resolution
    x = config.WORKSPACE_X[0] + (i + 0.5) * config.GRID_RESOLUTION
    y = config.WORKSPACE_Y[0] + (j + 0.5) * config.GRID_RESOLUTION
    z = config.WORKSPACE_Z[0] + (k + 0.5) * config.GRID_RESOLUTION

    world_pos = np.array([x, y, z])

    return world_pos


def is_valid_cell(i: int, j: int, k: int, config: AStarConfig) -> bool:
    """
    Check if grid indices are within valid bounds

    Arguments:
        i, j, k: Grid cell indices
        config: AStarConfig with grid dimensions

    Returns:
        True if cell is within grid bounds, False otherwise
    """

    valid = (0 <= i < config.nx and
             0 <= j < config.ny and
             0 <= k < config.nz)

    return valid


#### Example of how the convertion functions work.

In [143]:
# Test the conversion functions
print("Testing coordinate conversions...")

test_world = np.array([2.5, 3.7, 6.1])
test_grid = world_to_grid(test_world, config)
print(f"World {test_world} → Grid {test_grid}")

test_world_back = grid_to_world(test_grid[0], test_grid[1], test_grid[2], config)
print(f"Grid {test_grid} → World {test_world_back}")

print("\nCoordinate conversion functions working!")

Testing coordinate conversions...
World [2.5 3.7 6.1] → Grid (np.int64(6), np.int64(9), np.int64(15))
Grid (np.int64(6), np.int64(9), np.int64(15)) → World [2.6 3.8 6.2]

Coordinate conversion functions working!


### Collision Detection Fucntions for Sphere and Box Obstacles


In [144]:
# Sphere obstacle collision detection

def is_cell_in_sphere_obstacle(cell_world_pos: np.ndarray, sphere_center: np.ndarray, sphere_radius: float, safety_margin: float) -> bool:
    """
    Check if a cell (at its center) is inside a sphere obstacle (with safety margin)

    Arguments:
        cell_world_pos: numpy array [x, y, z] - center of the cell
        sphere_center: numpy array [x, y, z] - center of sphere obstacle
        sphere_radius: float - radius of the sphere
        safety_margin: float - extra clearance around obstacle

    Returns:
        True if cell is inside obstacle (should be marked as blocked), False otherwise

    Logic:
        Cell is inside if: distance(cell, sphere_center) < (radius + safety_margin)
    """

    # Calculate Euclidean distance from cell to sphere center
    distance = np.linalg.norm(cell_world_pos - sphere_center)

    # Check if within expanded obstacle radius
    threshold = sphere_radius + safety_margin
    is_inside = distance <= threshold

    return is_inside


In [145]:
# Box obstacle collision detection

def is_cell_in_box_obstacle(cell_world_pos: np.ndarray, box_min_corner: np.ndarray, box_max_corner: np.ndarray, safety_margin: float) -> bool:
    """
    Check if a cell (at its center) is inside a box obstacle (with safety margin)

    Arguments:
        cell_world_pos: numpy array [x, y, z] - center of the cell
        box_min_corner: numpy array [x, y, z] - minimum corner of box
        box_max_corner: numpy array [x, y, z] - maximum corner of box
        safety_margin: float - extra clearance around obstacle

    Returns:
        True if cell is inside obstacle (should be marked as blocked), False otherwise

    Logic:
        Box is defined by axis-aligned bounding box (AABB)
        Cell is inside if ALL of these conditions are true:
            expanded_min.x <= cell.x <= expanded_max.x  AND
            expanded_min.y <= cell.y <= expanded_max.y  AND
            expanded_min.z <= cell.z <= expanded_max.z

        Where:
            expanded_min = box_min_corner - safety_margin (expand box outward)
            expanded_max = box_max_corner + safety_margin (expand box outward)

    Example:
        Box from [3, 3, 3] to [7, 7, 7], safety_margin = 0.5
        Expanded box: [2.5, 2.5, 2.5] to [7.5, 7.5, 7.5]

        Cell at [5, 5, 5]: inside all bounds → inside (True)
        Cell at [2.7, 5, 5]: x=2.7 in [2.5, 7.5] → inside (True)
        Cell at [2.3, 5, 5]: x=2.3 < 2.5 → outside (False)
        Cell at [8, 5, 5]: x=8 > 7.5 → outside (False)
    """

    # Expand the box by safety margin
    expanded_min = box_min_corner - safety_margin
    expanded_max = box_max_corner + safety_margin

    # Extract cell coordinates
    x, y, z = cell_world_pos[0], cell_world_pos[1], cell_world_pos[2]

    # Check if cell is within expanded box on ALL three axes
    # Cell must be >= min AND <= max for all axes to be inside
    inside_x = (expanded_min[0] <= x <= expanded_max[0])
    inside_y = (expanded_min[1] <= y <= expanded_max[1])
    inside_z = (expanded_min[2] <= z <= expanded_max[2])

    # Cell is inside box only if inside on ALL axes
    is_inside = inside_x and inside_y and inside_z

    return is_inside


### Testing the Collision Detection Functions

In [146]:
# Test sphere collision detection
print("Testing sphere obstacle collision - ")

test_sphere_center = np.array([5.0, 5.0, 5.0])
test_sphere_radius = 1.5
test_safety = 0.5     # Taking higher value, 0.5 instead of 0.2, for testing

test_cases = [
    (np.array([5.0, 5.0, 5.0]), True, "at center"),
    (np.array([5.0, 5.0, 6.5]), True, "1.5m away (at radius boundary)"),
    (np.array([5.0, 5.0, 7.0]), True, "2.0m away (at safety boundary)"),
    (np.array([5.0, 5.0, 7.5]), False, "2.5m away (outside)"),
    (np.array([8.0, 8.0, 8.0]), False, "far away"),
]

print(f"\nSphere at {test_sphere_center}, radius={test_sphere_radius}, safety={test_safety}")
print(f"Total protected radius: {test_sphere_radius + test_safety}m\n")

all_correct = True
for pos, expected, description in test_cases:
    result = is_cell_in_sphere_obstacle(pos, test_sphere_center, test_sphere_radius, test_safety)
    status = "✓" if result == expected else "✗"
    all_correct = all_correct and (result == expected)
    print(f"  {status} Cell {description}: {result} (expected {expected})")

if all_correct:
    print("\nSphere collision detection working!")
else:
    print("\nSome test cases failed - check your logic!")


# Test box collision detection
print("\n\nTesting box obstacle collision - ")

test_box_min = np.array([3.0, 3.0, 3.0])
test_box_max = np.array([7.0, 7.0, 7.0])
test_safety = 0.5     # Taking higher value, 0.5 instead of 0.2, for testing

print(f"\nBox from {test_box_min} to {test_box_max}")
print(f"With safety margin {test_safety}, expanded to:")
print(f"  {test_box_min - test_safety} to {test_box_max + test_safety}\n")

test_cases = [
    (np.array([5.0, 5.0, 5.0]), True, "at center"),
    (np.array([3.0, 5.0, 5.0]), True, "at min boundary"),
    (np.array([7.0, 5.0, 5.0]), True, "at max boundary"),
    (np.array([2.7, 5.0, 5.0]), True, "just inside safety margin (x)"),
    (np.array([7.3, 5.0, 5.0]), True, "just inside safety margin (x)"),
    (np.array([2.3, 5.0, 5.0]), False, "outside safety margin (x)"),
    (np.array([7.7, 5.0, 5.0]), False, "outside safety margin (x)"),
    (np.array([5.0, 2.3, 5.0]), False, "outside safety margin (y)"),
    (np.array([5.0, 5.0, 2.3]), False, "outside safety margin (z)"),
    (np.array([1.0, 1.0, 1.0]), False, "far outside"),
    (np.array([9.0, 9.0, 9.0]), False, "far outside"),
    (np.array([2.7, 2.7, 2.7]), True, "corner inside safety"),
    (np.array([2.3, 2.7, 2.7]), False, "corner outside (one axis fails)"),
]

all_correct = True
for pos, expected, description in test_cases:
    result = is_cell_in_box_obstacle(pos, test_box_min, test_box_max, test_safety)
    status = "✓" if result == expected else "✗"
    all_correct = all_correct and (result == expected)
    print(f"  {status} Cell {description}: {result} (expected {expected})")

if all_correct:
    print("\nBox collision detection working!")
else:
    print("\nSome test cases failed!")
    print("Cell must be inside bounds on ALL three axes simultaneously")

Testing sphere obstacle collision - 

Sphere at [5. 5. 5.], radius=1.5, safety=0.5
Total protected radius: 2.0m

  ✓ Cell at center: True (expected True)
  ✓ Cell 1.5m away (at radius boundary): True (expected True)
  ✓ Cell 2.0m away (at safety boundary): True (expected True)
  ✓ Cell 2.5m away (outside): False (expected False)
  ✓ Cell far away: False (expected False)

Sphere collision detection working!


Testing box obstacle collision - 

Box from [3. 3. 3.] to [7. 7. 7.]
With safety margin 0.5, expanded to:
  [2.5 2.5 2.5] to [7.5 7.5 7.5]

  ✓ Cell at center: True (expected True)
  ✓ Cell at min boundary: True (expected True)
  ✓ Cell at max boundary: True (expected True)
  ✓ Cell just inside safety margin (x): True (expected True)
  ✓ Cell just inside safety margin (x): True (expected True)
  ✓ Cell outside safety margin (x): False (expected False)
  ✓ Cell outside safety margin (x): False (expected False)
  ✓ Cell outside safety margin (y): False (expected False)
  ✓ Cell outsi

### Building the Obstacle Grid for A* Algorithm
This function will mark all the cells that are in the obstacles. The algorithm will then, whn finding the route from start to goal position will know which cells are not to be traversed through.

In [147]:
# Building the complete obstacle grid (combines spheres and boxes)

def build_obstacle_grid(obstacles: List, config: AStarConfig) -> np.ndarray:
    """
    Build a 3D boolean grid marking which cells contain obstacles

    Arguments:
        obstacles: List of obstacle objects (from helpers.get_obstacles_hard())
                  Each obstacle is either:
                  - Sphere Obstacle: has .center and .radius attributes
                  - Box Obstacle: has .min_corner and .max_corner attributes
        config: AStarConfig with grid parameters

    Returns:
        obstacle_grid: 3D numpy boolean array of shape (nx, ny, nz)
                      True = cell contains obstacle, False = free space

    Algorithm:
        1. Create 3D array of False (all cells start as free)
        2. For each cell (i, j, k) in grid:
            a. Convert to world coordinates
            b. Check against each obstacle:
               - If sphere: use is_cell_in_sphere_obstacle()
               - If box: use is_cell_in_box_obstacle()
            c. If any obstacle contains cell, mark as True
        3. Return completed grid

    """

    print(f"Building obstacle grid...")
    print(f"  Grid size: {config.nx} x {config.ny} x {config.nz} = {config.nx * config.ny * config.nz:,} cells")
    print(f"  Obstacles: {len(obstacles)}")
    print(f"  Safety margin: {config.SAFETY_MARGIN}m")

    start_time = time.time()

    # Initialize grid with all False (all free space)
    obstacle_grid = np.zeros((config.nx, config.ny, config.nz), dtype=bool)

    # Counter for blocked cells
    blocked_count = 0

    # Loop through every cell in the grid
    for i in range(config.nx):
        for j in range(config.ny):
            for k in range(config.nz):

                # Get world position of cell center
                cell_world_pos = grid_to_world(i, j, k, config)

                # Check against each obstacle
                is_blocked = False

                for obstacle in obstacles:
                    # Check if obstacle is a sphere (has 'radius' attribute)
                    if hasattr(obstacle, 'radius'):
                        # Sphere obstacle
                        sphere_center = np.array(obstacle.center)
                        sphere_radius = obstacle.radius

                        if is_cell_in_sphere_obstacle(cell_world_pos, sphere_center, sphere_radius, config.SAFETY_MARGIN):
                            is_blocked = True
                            break  # No need to check other obstacles

                    else:
                        # Box obstacle (has 'min_corner' and 'max_corner')
                        box_min = np.array(obstacle.min_corner)
                        box_max = np.array(obstacle.max_corner)

                        if is_cell_in_box_obstacle(cell_world_pos, box_min, box_max, config.SAFETY_MARGIN):
                            is_blocked = True
                            break  # No need to check other obstacles

                # Mark cell if blocked
                if is_blocked:
                    obstacle_grid[i, j, k] = True
                    blocked_count += 1
    total_cells = config.nx * config.ny * config.nz
    free_cells = total_cells - blocked_count

    print(f"\n Obstacle grid built ")
    print(f"  Blocked cells: {blocked_count:,} ({blocked_count/total_cells*100:.1f}%)")
    print(f"  Free cells: {free_cells:,} ({free_cells/total_cells*100:.1f}%)")

    return obstacle_grid


# Build the obstacle grid for hard course
obstacle_grid = build_obstacle_grid(obstacles_hard, config)


Building obstacle grid...
  Grid size: 25 x 25 x 25 = 15,625 cells
  Obstacles: 8
  Safety margin: 0.2m

 Obstacle grid built 
  Blocked cells: 826 (5.3%)
  Free cells: 14,799 (94.7%)


### Heuristic Function
This function calculates the estimated cost from present cell to the goal cell position. Thus our optimizer works to reduce this value along with the cell transition value.

In [148]:
# Heuristic function for A*

def compute_heuristic(current_pos: np.ndarray, goal_pos: np.ndarray) -> float:
    """
    Compute the heuristic estimate of cost from current position to goal

    For A* to be optimal, the heuristic must be:
        h(n) <= true_cost(n, goal) for all nodes n

    Arguments:
        current_pos: numpy array [x, y, z] - current world position
        goal_pos: numpy array [x, y, z] - goal world position

    Returns:
        h_cost: float - estimated cost to reach goal

    Heuristic Choice: Euclidean Distance
        h(current, goal) = ||current - goal||
    """

    # Compute Euclidean distance (L2 norm)
    h_cost = np.linalg.norm(goal_pos - current_pos)

    return h_cost

print("\nHeuristic function working!")



Heuristic function working!


### Neighbor Function
This finds the neighbours of the current cell and calculates the cost function of moving to that cell.
The cost to move is sqareroot(no. of dimensions traversed) * resolution.


In [149]:
## TODO 6: Implement neighbor generation with 26-connectivity

def get_neighbors(current_cell: GridCell, obstacle_grid: np.ndarray, config: AStarConfig) -> List[Tuple[GridCell, float]]:
    """
    Generate all valid neighbors of the current cell

    26-Connected Grid:
        In 3D, each cell can move to 26 neighbors (3³ - 1 = 26)
        Movement in each axis: -1, 0, +1

    Movement Costs:
        - Straight (1 axis): cost = 1.0 × resolution
          Examples: (0,0,1), (1,0,0), (0,-1,0)

        - Diagonal 2D (2 axes): cost = √2 × resolution
          Examples: (1,1,0), (1,0,1), (0,1,-1)

        - Diagonal 3D (3 axes): cost = √3 × resolution
          Examples: (1,1,1), (1,-1,1), (-1,1,-1)

    Arguments:
        current_cell: GridCell object (current position in grid)
        obstacle_grid: 3D boolean array (True = obstacle)
        config: AStarConfig with grid parameters

    Returns:
        neighbors: List of tuples (neighbor_cell, movement_cost)
                  Only includes valid, collision-free neighbors

    Algorithm:
        1. Generate all 26 possible movements (dx, dy, dz) where each ∈ {-1, 0, 1}
        2. Skip (0, 0, 0) - that's the current cell
        3. For each movement:
            a. Calculate neighbor indices: (i+dx, j+dy, k+dz)
            b. Check if valid (within grid bounds)
            c. Check if free (not an obstacle)
            d. Calculate movement cost based on number of non-zero axes
            e. Create neighbor GridCell and add to list
        4. Return list of (neighbor, cost) tuples
    """

    neighbors = []

    # Try all 26 possible movements
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:

                # Skip the current cell (0, 0, 0)
                if dx == 0 and dy == 0 and dz == 0:
                    continue

                # Calculate neighbor grid coordinates
                ni = current_cell.i + dx
                nj = current_cell.j + dy
                nk = current_cell.k + dz

                # Check if neighbor is within grid bounds
                if not is_valid_cell(ni, nj, nk, config):
                    continue

                # Check if neighbor is blocked by obstacle
                if obstacle_grid[ni, nj, nk]:
                    continue

                # Calculate movement cost based on distance
                # Count how many axes we're moving along
                num_axes = (abs(dx) + abs(dy) + abs(dz))

                if num_axes == 1:
                    # Straight movement (one axis)
                    movement_cost = config.COST_STRAIGHT * config.GRID_RESOLUTION
                elif num_axes == 2:
                    # Diagonal 2D (two axes)
                    movement_cost = config.COST_DIAGONAL_2D * config.GRID_RESOLUTION
                else:  # num_axes == 3
                    # Diagonal 3D (three axes)
                    movement_cost = config.COST_DIAGONAL_3D * config.GRID_RESOLUTION

                # Create neighbor cell
                neighbor_world_pos = grid_to_world(ni, nj, nk, config)
                neighbor_cell = GridCell(ni, nj, nk,
                                        neighbor_world_pos[0],
                                        neighbor_world_pos[1],
                                        neighbor_world_pos[2])

                # Add to neighbors list
                neighbors.append((neighbor_cell, movement_cost))

    return neighbors

print("\nNeighbor generation working!")


Neighbor generation working!


### Main Function - Implementing the A* Algorithm
Below is the main fucntion that defines the core of the A* search algorithm.

In [150]:
# The Main A* Search Algorithm

def a_star_search(start_pos: np.ndarray, goal_pos: np.ndarray, obstacle_grid: np.ndarray, config: AStarConfig) -> Tuple[Optional[List[np.ndarray]], Dict]:
    """
    A* path planning algorithm in 3D grid

    Arguments:
        start_pos: numpy array [x, y, z] - start position in world coordinates
        goal_pos: numpy array [x, y, z] - goal position in world coordinates
        obstacle_grid: 3D boolean array - True = obstacle
        config: AStarConfig object

    Returns:
        path: List of world positions [x, y, z] from start to goal (or None if no path)
        info: Dictionary with statistics:
            - 'success': bool
            - 'nodes_expanded': int
            - 'path_length': float
            - 'planning_time': float

    A* Algorithm:
        1. Initialize open list (priority queue) with start node
        2. Initialize closed set (visited nodes)
        3. While open list not empty:
            a. Pop node with lowest f_cost from open list
            b. If node is goal, reconstruct path and return
            c. Add node to closed set
            d. For each neighbor:
                - Skip if in closed set
                - Calculate tentative g_cost
                - If neighbor not in open list OR found better path:
                    * Update neighbor costs and parent
                    * Add to open list
        4. If open list empty, no path exists
        We use (f_cost, counter, cell) to break ties consistently
    """

    print(f"\n{'='*60}")
    print(f" A* SEARCH")
    print(f"{'='*60}")
    print(f"Start: {start_pos}")
    print(f"Goal:  {goal_pos}")

    start_time = time.time()

    # Convert world positions to grid coordinates
    start_grid = world_to_grid(start_pos, config)
    goal_grid = world_to_grid(goal_pos, config)

    # Create start and goal cells
    start_world = grid_to_world(start_grid[0], start_grid[1], start_grid[2], config)
    goal_world = grid_to_world(goal_grid[0], goal_grid[1], goal_grid[2], config)

    start_cell = GridCell(start_grid[0], start_grid[1], start_grid[2],
                         start_world[0], start_world[1], start_world[2])
    goal_cell = GridCell(goal_grid[0], goal_grid[1], goal_grid[2],
                        goal_world[0], goal_world[1], goal_world[2])

    # Check if start or goal is blocked
    if obstacle_grid[start_grid[0], start_grid[1], start_grid[2]]:
        print("Start position is blocked by obstacle!")
        return None, {'success': False, 'nodes_expanded': 0, 'path_length': 0, 'planning_time': 0}

    if obstacle_grid[goal_grid[0], goal_grid[1], goal_grid[2]]:
        print("Goal position is blocked by obstacle!")
        return None, {'success': False, 'nodes_expanded': 0, 'path_length': 0, 'planning_time': 0}

    # Initialize start cell costs
    start_cell.g_cost = 0.0
    start_cell.h_cost = compute_heuristic(start_cell.world_position(), goal_cell.world_position())
    start_cell.f_cost = start_cell.g_cost + start_cell.h_cost
    start_cell.parent = None

    # Priority queue (open list): stores (f_cost, counter, cell)
    # Counter ensures consistent ordering when f_costs are equal
    open_list = []
    counter = 0  # Tie-breaker for equal f_costs
    heapq.heappush(open_list, (start_cell.f_cost, counter, start_cell))
    counter += 1

    # Closed set: stores grid positions (i, j, k) of visited nodes
    closed_set = set()

    # Open set: for quick lookup of cells in open list
    # Maps (i, j, k) -> cell
    open_set = {start_cell.position_tuple(): start_cell}

    # Statistics
    nodes_expanded = 0

    # Main A* loop
    while open_list:

        # Pop node with lowest f_cost
        current_f, _, current_cell = heapq.heappop(open_list)

        # Remove from open set
        current_pos_tuple = current_cell.position_tuple()
        if current_pos_tuple in open_set:
            del open_set[current_pos_tuple]

        # Check if reached goal
        if current_cell == goal_cell:
            # Success! Reconstruct path
            path = []
            node = current_cell
            while node is not None:
                path.append(node.world_position())
                node = node.parent
            path.reverse()  # Reverse to get start -> goal order

            planning_time = time.time() - start_time
            path_length = sum(np.linalg.norm(path[i+1] - path[i]) for i in range(len(path)-1))

            print(f"\n Path found!")
            print(f"   Nodes expanded: {nodes_expanded}")
            print(f"   Path length: {path_length:.2f}m")
            print(f"   Planning time: {planning_time:.3f}s")
            print(f"{'='*60}")

            return path, {
                'success': True,
                'nodes_expanded': nodes_expanded,
                'path_length': path_length,
                'planning_time': planning_time
            }

        # Add to closed set
        closed_set.add(current_pos_tuple)
        nodes_expanded += 1

        # Get neighbors
        neighbors = get_neighbors(current_cell, obstacle_grid, config)

        # Process each neighbor
        for neighbor_cell, move_cost in neighbors:

            neighbor_pos_tuple = neighbor_cell.position_tuple()

            # Skip if already visited
            if neighbor_pos_tuple in closed_set:
                continue

            # Calculate tentative g_cost
            tentative_g = current_cell.g_cost + move_cost

            # Check if neighbor is in open set
            if neighbor_pos_tuple in open_set:
                # Neighbor already in open list
                existing_neighbor = open_set[neighbor_pos_tuple]

                # Check if we found a better path
                if tentative_g < existing_neighbor.g_cost:
                    # Update with better path
                    existing_neighbor.g_cost = tentative_g
                    existing_neighbor.f_cost = existing_neighbor.g_cost + existing_neighbor.h_cost
                    existing_neighbor.parent = current_cell

                    # Re-add to priority queue with new f_cost
                    heapq.heappush(open_list, (existing_neighbor.f_cost, counter, existing_neighbor))
                    counter += 1

            else:
                # New neighbor - add to open list
                neighbor_cell.g_cost = tentative_g
                neighbor_cell.h_cost = compute_heuristic(neighbor_cell.world_position(), goal_cell.world_position())
                neighbor_cell.f_cost = neighbor_cell.g_cost + neighbor_cell.h_cost
                neighbor_cell.parent = current_cell

                # Add to open list and open set
                heapq.heappush(open_list, (neighbor_cell.f_cost, counter, neighbor_cell))
                counter += 1
                open_set[neighbor_pos_tuple] = neighbor_cell

    # If we get here, open list is empty - no path exists
    planning_time = time.time() - start_time

    print(f"\n No path found!")
    print(f"   Nodes expanded: {nodes_expanded}")
    print(f"   Planning time: {planning_time:.3f}s")
    print(f"{'='*60}")


    return None, {
        'success': False,
        'nodes_expanded': nodes_expanded,
        'path_length': 0,
        'planning_time': planning_time
    }


print(" A* algorithm implemented!")

 A* algorithm implemented!


### Testing (Debugging step)
These test cases were generated by Claude.ai to test if the codes had been running fine. This includes 3 test cases from simple to complex to test the code implementations.



In [151]:
## Test A* on a simple straight-line path (no obstacles in the way)

print("=" * 80)
print("TEST 1: Simple Straight-Line Path")
print("=" * 80)

# Simple test: straight line with no obstacles
test_start = np.array([1.0, 1.0, 5.0])
test_goal = np.array([9.0, 9.0, 5.0])

print(f"\nTest scenario:")
print(f"  Start: {test_start}")
print(f"  Goal: {test_goal}")
print(f"  Euclidean distance: {np.linalg.norm(test_goal - test_start):.2f}m")

# Run A*
path_simple, info_simple = a_star_search(test_start, test_goal, obstacle_grid, config)

if path_simple:
    print(f"\n✅ Success!")
    print(f"   Waypoints: {len(path_simple)}")
    print(f"   Path length: {info_simple['path_length']:.2f}m")
    print(f"   Efficiency: {np.linalg.norm(test_goal - test_start) / info_simple['path_length'] * 100:.1f}%")

    # Show first and last few waypoints
    print(f"\n   First 3 waypoints:")
    for i in range(min(3, len(path_simple))):
        print(f"     {i+1}. {path_simple[i]}")

    print(f"   ...")

    print(f"   Last 3 waypoints:")
    for i in range(max(0, len(path_simple)-3), len(path_simple)):
        print(f"     {i+1}. {path_simple[i]}")
else:
    print(f"\n❌ Failed to find path (unexpected!)")

print("\n" + "=" * 80)

TEST 1: Simple Straight-Line Path

Test scenario:
  Start: [1. 1. 5.]
  Goal: [9. 9. 5.]
  Euclidean distance: 11.31m

 A* SEARCH
Start: [1. 1. 5.]
Goal:  [9. 9. 5.]

 Path found!
   Nodes expanded: 20
   Path length: 11.31m
   Planning time: 0.022s

✅ Success!
   Waypoints: 21
   Path length: 11.31m
   Efficiency: 100.0%

   First 3 waypoints:
     1. [1. 1. 5.]
     2. [1.4 1.4 5. ]
     3. [1.8 1.8 5. ]
   ...
   Last 3 waypoints:
     19. [8.2 8.2 5. ]
     20. [8.6 8.6 5. ]
     21. [9. 9. 5.]



In [152]:
## Test A* with obstacles in the way

print("=" * 80)
print("TEST 2: Path with Obstacle Avoidance")
print("=" * 80)

# Test with start and goal that require avoiding obstacles
test_start_obs = np.array([2.0, 2.0, 2.0])
test_goal_obs = np.array([5.5, 7.0, 9.0])

print(f"\nTest scenario:")
print(f"  Start: {test_start_obs}")
print(f"  Goal: {test_goal_obs}")
print(f"  Obstacles: {len(obstacles_hard)}")
print(f"  Euclidean distance: {np.linalg.norm(test_goal_obs - test_start_obs):.2f}m")

# Run A*
path_obs, info_obs = a_star_search(test_start_obs, test_goal_obs, obstacle_grid, config)

if path_obs:
    print(f"\n✅ Success!")
    print(f"   Waypoints: {len(path_obs)}")
    print(f"   Path length: {info_obs['path_length']:.2f}m")
    print(f"   Planning time: {info_obs['planning_time']:.3f}s")
    print(f"   Nodes expanded: {info_obs['nodes_expanded']}")

    # Check collision
    has_collision, collision_info = helpers.check_path_collision(
        [gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(p[0], p[1], p[2])) for p in path_obs],
        obstacles_hard
    )

    if not has_collision:
        print(f"   ✅ Path is collision-free!")
    else:
        print(f"   ❌ Path has collisions (this shouldn't happen!)")
        print(f"      Collision: {collision_info}")
else:
    print(f"\n❌ Failed to find path")

print("\n" + "=" * 80)

TEST 2: Path with Obstacle Avoidance

Test scenario:
  Start: [2. 2. 2.]
  Goal: [5.5 7.  9. ]
  Obstacles: 8
  Euclidean distance: 9.29m

 A* SEARCH
Start: [2. 2. 2.]
Goal:  [5.5 7.  9. ]

 Path found!
   Nodes expanded: 1248
   Path length: 9.81m
   Planning time: 1.035s

✅ Success!
   Waypoints: 18
   Path length: 9.81m
   Planning time: 1.035s
   Nodes expanded: 1248
   ✅ Path is collision-free!



In [153]:
# Visualize the A* path in 3D

def visualize_astar_path(path: List[np.ndarray],
                         start_pos: np.ndarray,
                         goal_pos: np.ndarray,
                         obstacles: List,
                         title: str = "A* Path"):
    """
    Visualize A* path with obstacles
    """

    if path is None:
        print("No path to visualize!")
        return

    path_array = np.array(path)

    fig = go.Figure()

    # Add path
    fig.add_trace(go.Scatter3d(
        x=path_array[:, 0],
        y=path_array[:, 1],
        z=path_array[:, 2],
        mode='lines+markers',
        line=dict(color='blue', width=4),
        marker=dict(size=3, color='blue'),
        name='A* Path'
    ))

    # Add start
    fig.add_trace(go.Scatter3d(
        x=[start_pos[0]],
        y=[start_pos[1]],
        z=[start_pos[2]],
        mode='markers',
        marker=dict(size=10, color='green', symbol='diamond'),
        name='Start'
    ))

    # Add goal
    fig.add_trace(go.Scatter3d(
        x=[goal_pos[0]],
        y=[goal_pos[1]],
        z=[goal_pos[2]],
        mode='markers',
        marker=dict(size=10, color='red', symbol='diamond'),
        name='Goal'
    ))

    # Add obstacles
    for i, obs in enumerate(obstacles):
        if hasattr(obs, 'radius'):  # Sphere
            # Create sphere mesh
            u = np.linspace(0, 2 * np.pi, 20)
            v = np.linspace(0, np.pi, 20)
            x = obs.center[0] + obs.radius * np.outer(np.cos(u), np.sin(v))
            y = obs.center[1] + obs.radius * np.outer(np.sin(u), np.sin(v))
            z = obs.center[2] + obs.radius * np.outer(np.ones(np.size(u)), np.cos(v))

            fig.add_trace(go.Surface(
                x=x, y=y, z=z,
                opacity=0.3,
                colorscale=[[0, 'red'], [1, 'red']],
                showscale=False,
                name=f'Obstacle {i+1}' if i == 0 else None,
                showlegend=(i == 0)
            ))
        else:  # Box
            # Create box mesh (8 vertices)
            min_c = obs.min_corner
            max_c = obs.max_corner

            # Define 8 corners
            corners = [
                [min_c[0], min_c[1], min_c[2]],
                [max_c[0], min_c[1], min_c[2]],
                [max_c[0], max_c[1], min_c[2]],
                [min_c[0], max_c[1], min_c[2]],
                [min_c[0], min_c[1], max_c[2]],
                [max_c[0], min_c[1], max_c[2]],
                [max_c[0], max_c[1], max_c[2]],
                [min_c[0], max_c[1], max_c[2]],
            ]

            # Define 12 edges
            edges = [
                [0, 1], [1, 2], [2, 3], [3, 0],  # Bottom face
                [4, 5], [5, 6], [6, 7], [7, 4],  # Top face
                [0, 4], [1, 5], [2, 6], [3, 7]   # Vertical edges
            ]

            for edge in edges:
                p1, p2 = corners[edge[0]], corners[edge[1]]
                fig.add_trace(go.Scatter3d(
                    x=[p1[0], p2[0]],
                    y=[p1[1], p2[1]],
                    z=[p1[2], p2[2]],
                    mode='lines',
                    line=dict(color='red', width=4),
                    showlegend=False,
                    hoverinfo='skip'
                ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="X (m)",
            yaxis_title="Y (m)",
            zaxis_title="Z (m)",
            aspectmode='cube'
        ),
        width=900,
        height=900
    )

    fig.show()


# Visualize the obstacle avoidance path
if path_obs:
    print("Visualizing A* path with obstacles...\n")
    visualize_astar_path(path_obs, test_start_obs, test_goal_obs, obstacles_hard,
                        title="A* Path with Obstacle Avoidance")
    print("✅ Visualization complete!")

Visualizing A* path with obstacles...



✅ Visualization complete!


### Converting the A* Path to the Pose3 Path Configuration
This function changes only Yaw values according to the present and previous grid cell positions. Only if the goal position is provided, then only it changes the final orientation to the goal orientation else it always keeps the previous orientation when reaching the new grid point. This also applies in our current case when it has to just traverse through the 4 hoops and not specifically reach a target pose and position.


In [154]:
def grid_path_to_pose3_path(grid_path: List[np.ndarray], goal_pose: gtsam.Pose3 = None) -> List[gtsam.Pose3]:
    """
    Convert A* grid path to Pose3 trajectory

    Arguments:
        grid_path: List of [x, y, z] positions
        goal_pose: Optional target pose (if provided, uses its orientation at end)
    """
    pose_path = []

    for i in range(len(grid_path)):
        position = gtsam.Point3(grid_path[i][0], grid_path[i][1], grid_path[i][2])

        if i < len(grid_path) - 1:
            # Not last: direction-based orientation
            direction = grid_path[i+1] - grid_path[i]
            yaw = np.arctan2(direction[1], direction[0])
            pitch = 0.0
            roll = 0.0
            attitude = gtsam.Rot3.Ypr(yaw, pitch, roll)

        elif goal_pose is not None:
            # Last waypoint AND goal pose provided: use goal orientation
            attitude = goal_pose.rotation()

        else:
            # Last waypoint, no goal pose: use previous orientation
            if len(pose_path) > 0:
                attitude = pose_path[-1].rotation()
            else:
                attitude = gtsam.Rot3()  # Identity

        pose = gtsam.Pose3(attitude, position)
        pose_path.append(pose)

    return pose_path


## Implementing the A* Algorithm

In [155]:
# Defining the Function

def drone_racing_astar_with_obstacles(start: gtsam.Pose3, targets: List[gtsam.Pose3],obstacles: List,config: AStarConfig, obstacle_grid: np.ndarray) -> Tuple[List[gtsam.Pose3], Dict]:
    """
    Plan complete drone racing path using A* through multiple hoops

    Arguments:
        start: gtsam.Pose3 - starting pose
        targets: List[gtsam.Pose3] - hoop center targets (in order)
        obstacles: List of obstacle objects
        config: AStarConfig
        obstacle_grid: 3D boolean array of obstacles

    Returns:
        complete_path: List[gtsam.Pose3] - full path through all hoops
        stats: Dict with planning statistics

    Algorithm:
        1. For each hoop target:
            a. Run A* from current position to hoop center
            b. Convert grid path to Pose3 path
            c. Add hoop passage waypoints (helpers.pass_through_the_hoop)
            d. Append to complete path
            e. Update current position
        2. Track statistics (total time, segments, etc.)

    """

    complete_path = []
    current_position = start

    # Statistics
    total_planning_time = 0
    total_nodes_expanded = 0
    segment_count = 0
    all_segments_successful = True

    # Plan path segment for each hoop
    for idx, target in enumerate(targets):
        print(f"\n{'─'*60}")
        print(f"Planning segment {idx+1}/{len(targets)} to hoop at {target.translation()}")
        print(f"{'─'*60}")

        segment_start_time = time.time()

        # Run A* from current position to target
        grid_path, info = a_star_search(
            start_pos=np.array([current_position.translation()[0],
                               current_position.translation()[1],
                               current_position.translation()[2]]),
            goal_pos=np.array([target.translation()[0],
                              target.translation()[1],
                              target.translation()[2]]),
            obstacle_grid=obstacle_grid,
            config=config
        )

        segment_time = time.time() - segment_start_time

        # Check if segment planning succeeded
        if grid_path is None or not info['success']:
            print(f"Failed to find path to hoop {idx+1}")
            all_segments_successful = False
            break

        # Convert to Pose3 path
        segment_pose_path = grid_path_to_pose3_path(grid_path)

        # Add hoop passage waypoints
        helpers.pass_through_the_hoop(target, segment_pose_path)

        # Append to complete path (avoid duplicating first point)
        if len(complete_path) == 0:
            complete_path.extend(segment_pose_path)
        else:
            # Skip first point of segment (same as last point of previous segment)
            complete_path.extend(segment_pose_path[1:])

        # Update current position for next segment
        current_position = complete_path[-1]

        # Update statistics
        total_planning_time += segment_time
        total_nodes_expanded += info['nodes_expanded']
        segment_count += 1

        print(f" Segment {idx+1} complete:")
        print(f"   Waypoints: {len(segment_pose_path)}")
        print(f"   Path length: {info['path_length']:.2f}m")
        print(f"   Planning time: {segment_time:.3f}s")

    # Final statistics
    stats = {
        'success': all_segments_successful,
        'total_planning_time': total_planning_time,
        'total_nodes_expanded': total_nodes_expanded,
        'segments_completed': segment_count,
        'total_waypoints': len(complete_path)
    }

    print("\n" + "=" * 80)
    if all_segments_successful:
        print("✅ COMPLETE PATH PLANNED SUCCESSFULLY!")
        print(f"   Total waypoints: {len(complete_path)}")
        print(f"   Total planning time: {total_planning_time:.3f}s")
        print(f"   Total nodes expanded: {total_nodes_expanded:,}")
        print(f"   Segments: {segment_count}/{len(targets)}")
    else:
        print("❌ PATH PLANNING FAILED")
        print(f"   Completed segments: {segment_count}/{len(targets)}")
    print("=" * 80)

    return complete_path if all_segments_successful else None, stats


In [156]:
# Executing A* on the complete 8-obstacle hard course!
print("\n" + "=" * 80)
print("🏁 FINAL CHALLENGE: A* ON 8-OBSTACLE HARD COURSE 🏁\n")
print("🏁 DRONE RACING WITH A* - PLANNING PATH")
print("=" * 80)

# Run A* drone racing
astar_start_time = time.time()

astar_racing_path, astar_stats = drone_racing_astar_with_obstacles(
    start=start_pose,
    targets=targets,
    obstacles=obstacles_hard,
    config=config,
    obstacle_grid=obstacle_grid
)

astar_total_time = time.time() - astar_start_time

# Display results
if astar_racing_path:
    print("\n" + "🎉" * 40)
    print("SUCCESS! A* COMPLETED THE HARD COURSE!")
    print("🎉" * 40)

    # Validate collision-free
    print("\nValidating path...")
    has_collision, collision_details = helpers.check_path_collision( astar_racing_path, obstacles_hard)

    if not has_collision:
        print("✅ Path is COLLISION-FREE!")
        print("✅ All hoops reached!")
    else:
        print(f"⚠ Path has collisions: {collision_details}")

    # Path statistics
    print("\n📊 PATH STATISTICS:")
    print(f"   Total waypoints: {len(astar_racing_path)}")
    print(f"   Planning time: {astar_total_time:.3f}s")
    print(f"   Nodes expanded: {astar_stats['total_nodes_expanded']:,}")

    # Calculate path length
    path_length = 0
    for i in range(len(astar_racing_path) - 1):
        p1 = astar_racing_path[i].translation()
        p2 = astar_racing_path[i+1].translation()
        path_length += np.linalg.norm(np.array([p2[0]-p1[0], p2[1]-p1[1], p2[2]-p1[2]]))

    print(f"   Path length: {path_length:.2f}m")

else:
    print("\n❌ A* failed to complete the course")
    print(f"   Segments completed: {astar_stats['segments_completed']}/{len(targets)}")


🏁 FINAL CHALLENGE: A* ON 8-OBSTACLE HARD COURSE 🏁

🏁 DRONE RACING WITH A* - PLANNING PATH

────────────────────────────────────────────────────────────
Planning segment 1/4 to hoop at [7.29289322 5.         5.29289322]
────────────────────────────────────────────────────────────

 A* SEARCH
Start: [1. 3. 8.]
Goal:  [7.29289322 5.         5.29289322]

 Path found!
   Nodes expanded: 750
   Path length: 8.45m
   Planning time: 0.522s
 Segment 1 complete:
   Waypoints: 20
   Path length: 8.45m
   Planning time: 0.526s

────────────────────────────────────────────────────────────
Planning segment 2/4 to hoop at [ 7.5        13.70710678  1.5       ]
────────────────────────────────────────────────────────────

 A* SEARCH
Start: [8.70710678 5.         6.70710678]
Goal:  [ 7.5        13.70710678  1.5       ]

 Path found!
   Nodes expanded: 955
   Path length: 8.33m
   Planning time: 0.946s
 Segment 2 complete:
   Waypoints: 17
   Path length: 8.33m
   Planning time: 0.950s

────────────────

## Visualizing the Results

In [157]:
# Visualize the complete A* racing path with hoops and obstacles

if astar_racing_path:
    print("\nVisualizing A* racing path...\n")

    fig = helpers.drone_racing_path_with_obstacles(
        hoops=hoops,
        start=start_pose,
        path=astar_racing_path,
        obstacles=obstacles_hard,
    )

    fig

    print("✅ Visualization complete!")
    print("\n🎮 Interactive controls:")
    print("   - Rotate: Click and drag")
    print("   - Zoom: Scroll wheel")
    print("   - Pan: Right-click and drag")
else:
    print("⚠ No path to visualize")


Visualizing A* racing path...



✅ Visualization complete!

🎮 Interactive controls:
   - Rotate: Click and drag
   - Zoom: Scroll wheel
   - Pan: Right-click and drag


### Smoothing

In [166]:
## Path Smoothing: Remove redundant waypoints while preserving hoop passages

def smooth_path_shortcut_with_hoops(path: List[gtsam.Pose3],
                                    obstacles: List,
                                    hoops: List[gtsam.Pose3],
                                    max_iterations: int = 100) -> List[gtsam.Pose3]:
    """
    Smooth path by removing unnecessary intermediate waypoints
    WHILE PRESERVING HOOP PASSAGE WAYPOINTS

    Algorithm (Hoop-Aware Shortcutting):
        1. Identify which waypoints are near hoops (must be kept)
        2. Try to connect non-adjacent waypoints directly
        3. If direct connection is collision-free AND doesn't skip hoops, remove intermediate
        4. Repeat until no more shortcuts possible

    This reduces waypoint count without breaking hoop passages

    Arguments:
        path: List[gtsam.Pose3] - original jagged path
        obstacles: List of obstacles to check collisions
        hoops: List[gtsam.Pose3] - hoop positions to preserve
        max_iterations: maximum shortcutting attempts

    Returns:
        smoothed_path: List[gtsam.Pose3] - smoother path with fewer waypoints
    """

    if len(path) <= 2:
        return path

    print(f"\n{'='*60}")
    print(f"🔧 PATH SMOOTHING (HOOP-AWARE)")
    print(f"{'='*60}")
    print(f"Original path: {len(path)} waypoints")
    print(f"Hoops to preserve: {len(hoops)}")

    # Step 1: Identify hoop waypoints (must NOT be removed)
    hoop_waypoint_indices = set()
    hoop_threshold = 0.3  # Within 0.3m of hoop = hoop waypoint

    for hoop_idx, hoop in enumerate(hoops):
        hoop_pos = np.array([hoop.translation()[0],
                            hoop.translation()[1],
                            hoop.translation()[2]])

        # Find closest waypoint to this hoop
        min_dist = float('inf')
        closest_idx = -1

        for wp_idx, waypoint in enumerate(path):
            wp_pos = np.array([waypoint.translation()[0],
                              waypoint.translation()[1],
                              waypoint.translation()[2]])
            dist = np.linalg.norm(wp_pos - hoop_pos)

            if dist < min_dist:
                min_dist = dist
                closest_idx = wp_idx

        # Mark this waypoint as protected if it's close to hoop
        if min_dist < hoop_threshold:
            hoop_waypoint_indices.add(closest_idx)
            print(f"   Protected waypoint {closest_idx} (hoop {hoop_idx+1}, dist={min_dist:.2f}m)")

    print(f"   Total protected waypoints: {len(hoop_waypoint_indices)}")

    # Step 2: Shortcut smoothing (avoiding protected waypoints)
    smoothed = list(path)  # Copy path

    # Adjust protected indices as we remove waypoints
    protected_indices = sorted(hoop_waypoint_indices)

    iteration = 0
    improved = True

    while improved and iteration < max_iterations:
        improved = False
        i = 0

        while i < len(smoothed) - 2:
            # Check if we're trying to remove a protected waypoint
            if (i + 1) in protected_indices:
                # Can't remove this waypoint - it's a hoop passage point
                i += 1
                continue

            # Try to shortcut from waypoint i to i+2 (skip i+1)
            pos_i = np.array([smoothed[i].translation()[0],
                             smoothed[i].translation()[1],
                             smoothed[i].translation()[2]])
            pos_i2 = np.array([smoothed[i+2].translation()[0],
                              smoothed[i+2].translation()[1],
                              smoothed[i+2].translation()[2]])

            # Check collision on direct segment
            has_collision = helpers.check_segment_collision(pos_i, pos_i2, obstacles)

            if not has_collision:
                # Can shortcut! Remove intermediate waypoint
                smoothed.pop(i+1)
                improved = True

                # Update protected indices (shift down by 1)
                protected_indices = [idx if idx <= i else idx - 1 for idx in protected_indices]

                # Don't increment i, check again from same position
            else:
                i += 1

        iteration += 1

    print(f"\nSmoothed path: {len(smoothed)} waypoints")
    print(f"Removed: {len(path) - len(smoothed)} waypoints ({(len(path) - len(smoothed))/len(path)*100:.1f}%)")
    print(f"Protected waypoints preserved: {len(protected_indices)}")
    print(f"Iterations: {iteration}")
    print(f"{'='*60}")

    return smoothed


# Apply hoop-aware smoothing to A* path
if astar_racing_path:
    print("\nApplying hoop-aware path smoothing...")

    astar_path_smoothed = smooth_path_shortcut_with_hoops(
        path=astar_racing_path,
        obstacles=obstacles_hard,
        hoops=hoops,  # Pass hoops!
        max_iterations=50
    )

    # Calculate path lengths
    def compute_path_length(path):
        length = 0
        for i in range(len(path) - 1):
            p1 = path[i].translation()
            p2 = path[i+1].translation()
            length += np.linalg.norm(np.array([p2[0]-p1[0], p2[1]-p1[1], p2[2]-p1[2]]))
        return length

    original_length = compute_path_length(astar_racing_path)
    smoothed_length = compute_path_length(astar_path_smoothed)

    print(f"\n📊 Smoothing Results:")
    print(f"   Original: {len(astar_racing_path)} waypoints, {original_length:.2f}m")
    print(f"   Smoothed: {len(astar_path_smoothed)} waypoints, {smoothed_length:.2f}m")
    print(f"   Path shortened by: {original_length - smoothed_length:.2f}m ({(original_length - smoothed_length)/original_length*100:.1f}%)")

    # Verify still collision-free
    has_collision_smooth, _ = helpers.check_path_collision(astar_path_smoothed, obstacles_hard)
    print(f"   Collision-free: {' Yes' if not has_collision_smooth else ' No'}")

    # IMPORTANT: Verify all hoops are still reached
    print(f"\n Hoop Verification:")
    hoops_reached = 0
    for hoop_idx, hoop in enumerate(hoops):
        hoop_pos = np.array([hoop.translation()[0],
                            hoop.translation()[1],
                            hoop.translation()[2]])

        # Check if any waypoint is close to this hoop
        min_dist = float('inf')
        for waypoint in astar_path_smoothed:
            wp_pos = np.array([waypoint.translation()[0],
                              waypoint.translation()[1],
                              waypoint.translation()[2]])
            dist = np.linalg.norm(wp_pos - hoop_pos)
            min_dist = min(min_dist, dist)

        if min_dist < 0.5:  # Within 0.5m = passed through
            hoops_reached += 1
            print(f"  Hoop {hoop_idx+1}: closest waypoint {min_dist:.2f}m")
        else:
            print(f"  Hoop {hoop_idx+1}: closest waypoint {min_dist:.2f}m (TOO FAR!)")

    print(f"\n   Total hoops reached: {hoops_reached}/{len(hoops)}")

    if hoops_reached == len(hoops):
        print("\nPath smoothing complete - ALL HOOPS PRESERVED!")
    else:
        print(" Consider using original path or adjusting protection threshold")

else:
    print(" Yayay! No path to smooth")


Applying hoop-aware path smoothing...

🔧 PATH SMOOTHING (HOOP-AWARE)
Original path: 70 waypoints
Hoops to preserve: 4
   Protected waypoint 18 (hoop 1, dist=0.00m)
   Protected waypoint 34 (hoop 2, dist=0.00m)
   Protected waypoint 55 (hoop 3, dist=0.00m)
   Protected waypoint 68 (hoop 4, dist=0.00m)
   Total protected waypoints: 4

Smoothed path: 13 waypoints
Removed: 57 waypoints (81.4%)
Protected waypoints preserved: 4
Iterations: 2

📊 Smoothing Results:
   Original: 70 waypoints, 49.53m
   Smoothed: 13 waypoints, 38.64m
   Path shortened by: 10.89m (22.0%)
   Collision-free: ✅ Yes

🎯 Hoop Verification:
   ✅ Hoop 1: closest waypoint 0.00m
   ✅ Hoop 2: closest waypoint 0.00m
   ✅ Hoop 3: closest waypoint 0.00m
   ✅ Hoop 4: closest waypoint 0.00m

   Total hoops reached: 4/4

✅ Path smoothing complete - ALL HOOPS PRESERVED!


In [188]:
# Overlaying both paths in one plot

if astar_racing_path and astar_path_smoothed:
    print("\nCreating overlay comparison...\n")

    # First, show smoothed path (it will be the base)
    helpers.drone_racing_path_with_obstacles(
        hoops=hoops,
        start=start_pose,
        path=astar_path_smoothed,
        obstacles=obstacles_hard,
       )

    # Note: The helper function doesn't return the figure, so we can't overlay
    # But showing them sequentially is actually clearer for comparison!

    print("\n" + "-"*80)

    # Then show original path for comparison
    helpers.drone_racing_path_with_obstacles(
        hoops=hoops,
        start=start_pose,
        path=astar_racing_path,
        obstacles=obstacles_hard,
        )

    print("\n✅ Both visualizations shown - compare the waypoint density!")

else:
    print("⚠ No path available")


Creating overlay comparison...




--------------------------------------------------------------------------------



✅ Both visualizations shown - compare the waypoint density!
